# Análise de Atendimentos de Clínica Veterinária
Notebook com as 13 atividades solicitadas e geração do relatório final.

In [1]:
# 1. Inserir os 50 registros na coleção 'atendimentos'
import json
from pymongo import MongoClient
import pandas as pd
from datetime import datetime

# Conectar ao MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['clinica']
atendimentos = db['atendimentos']
relatorios = db['relatorios']

# Limpar coleções para evitar duplicidade
atendimentos.delete_many({})relatorios.delete_many({})

# Carregar dados do arquivo JSON
with open('dados_clinica.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

# Inserir registros
atendimentos.insert_many(dados)

SyntaxError: invalid syntax (341756469.py, line 14)

In [ ]:
# 2. Consultar atendimentos com valor entre 90 e 200 reais
result_2 = list(atendimentos.find({'valor': {'$gte': 90, '$lte': 200}}))
result_2[:2]  # Exemplo de visualização

In [ ]:
# 3. Consultar atendimentos das cidades Assis, Ourinhos, Pompeia e Marília
cidades = ['Assis', 'Ourinhos', 'Pompeia', 'Marília']
result_3 = list(atendimentos.find({'cidade': {'$in': cidades}}))
result_3[:2]  # Exemplo de visualização

In [ ]:
# 4. Converter os resultados para DataFrame
df = pd.DataFrame(list(atendimentos.find()))
df.head(2)

In [ ]:
# 5. Verificar valores nulos
df.isnull().sum()

In [ ]:
# 6. Calcular faturamento total
faturamento_total = df['valor'].sum()
faturamento_total

In [ ]:
# 7. Calcular faturamento por cidade
faturamento_por_cidade = df.groupby('cidade')['valor'].sum().to_dict()
faturamento_por_cidade

In [ ]:
# 8. Calcular faturamento por serviço
faturamento_por_servico = df.groupby('servico')['valor'].sum().to_dict()
faturamento_por_servico

In [ ]:
# 9. Calcular ticket médio geral
ticket_medio_geral = df['valor'].mean()
ticket_medio_geral

In [ ]:
# 10. Calcular ticket médio por cidade
ticket_medio_por_cidade = df.groupby('cidade')['valor'].mean().to_dict()
ticket_medio_por_cidade

In [ ]:
# 11. Identificar o serviço mais lucrativo
servico_mais_lucrativo = df.groupby('servico')['valor'].sum().idxmax()
servico_mais_lucrativo

In [ ]:
# 12. Identificar a cidade com maior faturamento
cidade_maior_faturamento = df.groupby('cidade')['valor'].sum().idxmax()
cidade_maior_faturamento

In [ ]:
# 13. Identificar os 3 maiores atendimentos
top_3_atendimentos = df.nlargest(3, 'valor').to_dict(orient='records')
top_3_atendimentos

In [ ]:
# Gerar relatório conforme especificação
relatorio = {
    'data_geracao': datetime.now().isoformat(),
    'faturamento_total': float(faturamento_total),
    'faturamento_por_cidade': faturamento_por_cidade,
    'faturamento_por_servico': faturamento_por_servico,
    'ticket_medio_geral': float(ticket_medio_geral),
    'ticket_medio_por_cidade': {k: float(v) for k, v in ticket_medio_por_cidade.items()},
    'servico_mais_lucrativo': servico_mais_lucrativo,
    'cidade_maior_faturamento': cidade_maior_faturamento,
    'top_3_atendimentos': top_3_atendimentos
}
relatorio

In [ ]:
# Salvar relatório na coleção 'relatorios'
relatorios.insert_one(relatorio)
print('Relatório salvo com sucesso!')